# 01 — CV Data Prep: unify CarDD + VehiDE into one YOLO dataset

**Run on Kaggle (CPU is fine).** Attach these two datasets as inputs:
- `gabrielfcarvalho/cardd-with-yolo-annotations-images-labels` — CarDD, already YOLO (`train|val|test/images + labels`, 6 classes)
- `hendrichscullen/vehide-dataset-automatic-vehicle-damage-detection` — VehiDE, VIA-style polygon JSON (`0Train_via_annos.json`, `0Val_via_annos.json`, images under `image/image/`)

**Verified facts this notebook is built on** (inspected 2026-07):
- CarDD `data.yaml` class order = `['dent','scratch','crack','glass shatter','lamp broken','tire flat']` → identical to unified ids 0–5, so CarDD label files pass through **unchanged**.
- VehiDE regions are `{all_x:[...], all_y:[...], class:<vi>}` polygons with **no image sizes** (read via PIL headers) and Vietnamese classes:
  `tray_son`(paint scratch), `mop_lom`(dent), `vo_kinh`(broken glass), `be_den`(broken lamp), `rach`(torn), `thung`(puncture), `mat_bo_phan`(missing part).

## Unified 8-class damage schema
| id | class | CarDD source | VehiDE source |
|----|-------|--------------|---------------|
| 0 | dent | dent | mop_lom |
| 1 | scratch | scratch | tray_son |
| 2 | crack | crack | rach (torn ≈ linear rupture) |
| 3 | glass_shatter | glass shatter | vo_kinh |
| 4 | lamp_broken | lamp broken | be_den |
| 5 | tire_flat | tire flat | — |
| 6 | punctured | — | thung |
| 7 | missing_part | — | mat_bo_phan |

Nothing is dropped: VehiDE's two extra damage types get their own classes rather than being silently discarded. Any *unknown* class halts the merge.

In [ ]:
import glob, json, shutil, sys
from pathlib import Path
from collections import Counter

INPUT = Path("/kaggle/input")
print("mounted inputs:", [p.name for p in INPUT.iterdir()] if INPUT.exists() else "NONE")

# Kaggle may mount datasets at /kaggle/input/<slug> or nested under
# /kaggle/input/datasets/<owner>/<slug> — search recursively for marker files.
def locate(marker_glob, extra_pred, what):
    for hit in sorted(glob.glob(f"/kaggle/input/**/{marker_glob}", recursive=True)):
        root = Path(hit).parent
        if extra_pred(root):
            return root
    raise AssertionError(f"could not locate {what} under /kaggle/input — attach the dataset")

CARDD  = locate("data.yaml", lambda r: (r/"train"/"images").exists(), "CarDD (YOLO)")
VEHIDE = locate("0Train_via_annos.json", lambda r: True, "VehiDE (VIA)")
OUT    = Path("/kaggle/working/combined")
print("CarDD root: ", CARDD)
print("VehiDE root:", VEHIDE)

UNIFIED = ["dent","scratch","crack","glass_shatter","lamp_broken","tire_flat","punctured","missing_part"]
U = {n:i for i,n in enumerate(UNIFIED)}
VEHIDE_REMAP = {
    "mop_lom":"dent", "tray_son":"scratch", "rach":"crack",
    "vo_kinh":"glass_shatter", "be_den":"lamp_broken",
    "thung":"punctured", "mat_bo_phan":"missing_part",
}

for split in ["train","val"]:
    (OUT/"images"/split).mkdir(parents=True, exist_ok=True)
    (OUT/"labels"/split).mkdir(parents=True, exist_ok=True)
print("output dirs ready:", OUT)

### 1. CarDD — copy through (ids already match unified 0–5)
Verify the class order from its own `data.yaml` first; CarDD `test` split is folded into `val`
(we carve our own held-out half in notebook 03).

In [ ]:
import yaml
cardd_names = yaml.safe_load(open(CARDD/"data.yaml"))["names"]
expected = ['dent','scratch','crack','glass shatter','lamp broken','tire flat']
assert cardd_names == expected, f"CarDD class order changed! {cardd_names}"
print("CarDD class order verified ✓")

def copy_cardd(src_split, dst_split):
    imgs = sorted(glob.glob(str(CARDD/src_split/"images"/"*")))
    n = 0
    for img in imgs:
        stem = Path(img).stem
        shutil.copy(img, OUT/"images"/dst_split/f"cardd_{Path(img).name}")
        lbl = CARDD/src_split/"labels"/f"{stem}.txt"
        txt = lbl.read_text() if lbl.exists() else ""
        (OUT/"labels"/dst_split/f"cardd_{stem}.txt").write_text(txt)
        n += 1
    print(f"CarDD {src_split} -> {dst_split}: {n} images")
    return n

n_cardd  = copy_cardd("train", "train")
n_cardd += copy_cardd("val",   "val")
n_cardd += copy_cardd("test",  "val")   # folded; notebook 03 re-splits val deterministically

### 2. VehiDE — VIA polygons → YOLO boxes (image sizes via PIL headers)

In [ ]:
from PIL import Image

IMG_ROOT = VEHIDE/"image"/"image"

def convert_vehide(anno_file, dst_split):
    data = json.load(open(VEHIDE/anno_file))
    unknown = Counter()
    n_img = n_box = n_missing = 0
    for fname, entry in data.items():
        src = IMG_ROOT/fname
        if not src.exists():
            n_missing += 1
            continue
        regions = entry.get("regions") or []
        try:
            with Image.open(src) as im:
                W, H = im.size          # header read only — fast
        except Exception:
            n_missing += 1
            continue
        lines = []
        for r in regions:
            cls = str(r.get("class","")).strip().lower()
            if cls not in VEHIDE_REMAP:
                unknown[cls] += 1
                continue
            xs, ys = r.get("all_x"), r.get("all_y")
            if not xs or not ys:
                continue
            x0, x1 = max(0,min(xs)), min(W,max(xs))
            y0, y1 = max(0,min(ys)), min(H,max(ys))
            if x1<=x0 or y1<=y0:
                continue
            cx, cy = (x0+x1)/2/W, (y0+y1)/2/H
            bw, bh = (x1-x0)/W, (y1-y0)/H
            lines.append(f"{U[VEHIDE_REMAP[cls]]} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
            n_box += 1
        shutil.copy(src, OUT/"images"/dst_split/f"vehide_{fname}")
        (OUT/"labels"/dst_split/f"vehide_{Path(fname).stem}.txt").write_text("\n".join(lines))
        n_img += 1
    print(f"VehiDE {anno_file} -> {dst_split}: {n_img} images, {n_box} boxes, {n_missing} missing files")
    if unknown:
        raise AssertionError(f"UNKNOWN VehiDE classes (extend VEHIDE_REMAP): {dict(unknown)}")
    return n_img

n_vehide  = convert_vehide("0Train_via_annos.json", "train")
n_vehide += convert_vehide("0Val_via_annos.json",   "val")

### 3. `data.yaml` + class balance report

In [ ]:
yaml_txt = f'''path: {OUT}
train: images/train
val: images/val
nc: {len(UNIFIED)}
names: {UNIFIED}
'''
(OUT/"data.yaml").write_text(yaml_txt)
print(yaml_txt)

cnt, per_split = Counter(), {"train":Counter(), "val":Counter()}
for split in ["train","val"]:
    for lf in glob.glob(str(OUT/"labels"/split/"*.txt")):
        for ln in open(lf):
            if ln.strip():
                c = UNIFIED[int(ln.split()[0])]
                cnt[c] += 1; per_split[split][c] += 1

print(f"{'class':16s} {'train':>7s} {'val':>7s} {'total':>7s}")
for k in UNIFIED:
    print(f"{k:16s} {per_split['train'][k]:7d} {per_split['val'][k]:7d} {cnt[k]:7d}")
n_tr = len(glob.glob(str(OUT/'images/train/*')))
n_va = len(glob.glob(str(OUT/'images/val/*')))
print(f"\nimages: train={n_tr} val={n_va} total={n_tr+n_va} (cardd={n_cardd}, vehide={n_vehide})")

report = {"unified_classes": UNIFIED,
          "instance_counts": {k: cnt[k] for k in UNIFIED},
          "images": {"train": n_tr, "val": n_va}}
json.dump(report, open("/kaggle/working/dataset_report.json","w"), indent=2)
print("\nwrote /kaggle/working/dataset_report.json")

### Output
`/kaggle/working/combined/` — one YOLO dataset (`images/`, `labels/`, `data.yaml`) on the unified 8-class schema, plus `dataset_report.json` with the real class balance (record it in `docs/ARCHITECTURE.md`).

Notebook 02 consumes this kernel's output directly (attached as a kernel source) — no manual dataset creation needed when running via the API chain.